# Phase 7 — Evaluation (RAGAS faithfulness, TruLens groundedness, custom citation check)

Runs 4 questions (2 grounded, 1 off-topic, 1 topically-close-but-ungrounded)
through the Phase 5 graph, then scores each answer three independent ways.
See `learning/09_evaluation/01_ragas_trulens_citation_check.md` for why.

## Build the collection + graph

Same recipe as notebook 11. Needs `backend/.env` with `OPENAI_API_KEY`.

In [1]:
import logging
import os
import sys
from pathlib import Path

logging.getLogger("numexpr").setLevel(logging.WARNING)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

sys.path.insert(0, str(Path.cwd().parent))

from generate_sample_pdf import generate

from src.graph import build_graph
from src.llm import get_llm
from src.retrieval import RetrievalPipeline

pdf_path = generate()
pipeline = RetrievalPipeline(storage_path=Path("data/qdrant"))
pipeline.ingest(pdf_path)

llm = get_llm("openai")
graph = build_graph(pipeline, llm)

questions = [
    ("grounded_citations", "How is citation accuracy measured in this paper?"),
    ("grounded_chunking", "How are documents split into chunks?"),
    ("offtopic", "What is the capital city of France?"),
    ("ungrounded_funding", "What funding source supported this research?"),
]

results = {}
for name, question in questions:
    results[name] = graph.invoke({"question": question})
    print(f"{name}: grounded={results[name]['grounded']}, pages={[c['page_label'] for c in results[name]['chunks']]}")

/Users/rtlecl910010/Desktop/Projects/Project/papermind/backend/.venv/lib/python3.11/site-packages/huggingface_hub/utils/tqdm.py:176: UserWarning: Cannot enable progress bars: environment variable `HF_HUB_DISABLE_PROGRESS_BARS=1` is set and has priority.
  warnings.warn(


grounded_citations: grounded=False, pages=['1', '4']


grounded_chunking: grounded=True, pages=['3', '2']


offtopic: grounded=False, pages=['3', '1']


ungrounded_funding: grounded=False, pages=['4', '1']


## RAGAS faithfulness

Real OpenAI calls (gpt-4o-mini) per question, graded by RAGAS's own
faithfulness prompt.

In [2]:
from openai import OpenAI as OpenAIClient
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

ragas_llm = llm_factory("gpt-4o-mini", client=OpenAIClient())
faithfulness_metric = Faithfulness(llm=ragas_llm)

ragas_scores = {}
for name, result in results.items():
    contexts = [chunk["text"] for chunk in result["chunks"]]
    score = await faithfulness_metric.ascore(
        user_input=result["question"], response=result["answer"], retrieved_contexts=contexts
    )
    ragas_scores[name] = score.value
    print(f"{name}: ragas faithfulness = {score.value}")

grounded_citations: ragas faithfulness = 0.0


grounded_chunking: ragas faithfulness = 1.0


offtopic: ragas faithfulness = 1.0


ungrounded_funding: ragas faithfulness = 0.0


## TruLens groundedness

Independent library, independent prompt, same job: does the answer follow
from the retrieved context?

In [3]:
from trulens.providers.openai import OpenAI as TruOpenAI

tru_provider = TruOpenAI(model_engine="gpt-4o-mini")

trulens_scores = {}
for name, result in results.items():
    source = "\n\n".join(chunk["text"] for chunk in result["chunks"])
    score, reasons = tru_provider.groundedness_measure_with_cot_reasons(
        source=source, statement=result["answer"]
    )
    trulens_scores[name] = score
    print(f"{name}: trulens groundedness = {score}")

2026-11-25 19:44:17,420 - INFO - setup plugin alembic.autogenerate.schemas


2026-11-25 19:44:17,420 - INFO - setup plugin alembic.autogenerate.tables


2026-11-25 19:44:17,421 - INFO - setup plugin alembic.autogenerate.types


2026-11-25 19:44:17,421 - INFO - setup plugin alembic.autogenerate.constraints


2026-11-25 19:44:17,421 - INFO - setup plugin alembic.autogenerate.defaults


2026-11-25 19:44:17,421 - INFO - setup plugin alembic.autogenerate.comments


... (2 outputs left out)

grounded_citations: trulens groundedness = 1.0


grounded_chunking: trulens groundedness = 0.8333333333333333


offtopic: trulens groundedness = 1.0


ungrounded_funding: trulens groundedness = 1.0


## Custom citation-precision check

Code-only, no LLM call: of the page numbers the answer cites, what
fraction were actually retrieved?

In [4]:
from src.evaluation import citation_precision

citation_scores = {}
for name, result in results.items():
    citation_scores[name] = citation_precision(result["answer"], result["chunks"])
    print(f"{name}: citation precision = {citation_scores[name]}")

grounded_citations: citation precision = None
grounded_chunking: citation precision = 1.0
offtopic: citation precision = None
ungrounded_funding: citation precision = None


## Summary table

All three scores side by side, read honestly: do they agree? Where do
they diverge, and does that match what tasks #13/#14 already observed by
hand?

In [5]:
print(f"{'question':<22} {'ragas':>8} {'trulens':>8} {'citation':>10}")
for name, _ in questions:
    print(
        f"{name:<22} {ragas_scores[name]:>8.2f} {trulens_scores[name]:>8.2f} "
        f"{str(citation_scores[name]):>10}"
    )

question                  ragas  trulens   citation
grounded_citations         0.00     1.00       None
grounded_chunking          1.00     0.83        1.0
offtopic                   1.00     1.00       None
ungrounded_funding         0.00     1.00       None


## Next

Phase 8 adds observability (LangSmith, JSON logging) so these kinds of
scores/traces can be captured automatically on every real request, not
just recomputed by hand in a notebook.